# PE6201 A2 — LI ZIHAO loop smoke test
This notebook demonstrates the provider-neutral manual loop without using an API key. Production code lives in `src/`; the notebook is only an experiment and demo surface.

In [ ]:
import json
from src.agent.loop import run_agent
from src.schemas import GuardConfig, ModelResponse, ToolResult

In [ ]:
class DemoBackend:
    name = 'scripted-demo'
    def __init__(self):
        self.turn = 0
    def generate(self, messages, *, model, temperature=0.0):
        self.turn += 1
        if self.turn == 1:
            payload = {'type': 'action_block', 'reasoning_summary': 'Read claim', 'actions': [{'call_id': 't01-c01', 'tool': 'get_claim', 'args': {'case_id': 'CLM-8842'}}]}
        else:
            payload = {'type': 'final', 'final': {'decision': 'approve_in_principle', 'trigger': None, 'missing': None, 'escalate_to': None, 'line_dispositions': [], 'approved_total': 0, 'refused_total': 0, 'evidence': ['CLM-8842']}}
        return ModelResponse(json.dumps(payload), 10, 5, model, cost_usd=0.001)

In [ ]:
tools = {'get_claim': lambda case_id: ToolResult(True, {'claim_id': case_id})}
result = run_agent('CLM-8842', backend=DemoBackend(), model='scripted-demo', parallel_tools=True, autonomy='confirm', max_steps=4, budget_usd=0.05, guard_config=GuardConfig(), tool_registry=tools)
result.to_dict()